In [15]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts


 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [16]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


#  Intro
 ML – Claims High-Cost Risk Model

 **Goal:** Train and evaluate Spark ML models to predict *high-cost claims* using
 the curated Gold feature table `ft_claims_risk_split`.

 Label: `Is_High_Cost` (0 = normal, 1 = high cost)

 Steps:
 1. Load Gold feature table
 2. Clean + train/test split
 3. Build Spark ML pipeline (index + encode + assemble)
 4. Train several models (LR, RF, GBT)
 5. Persist the best model
 6. Score sample claims with high-cost probability


# Cell 1 — MLflow setup for high-cost model

In [17]:

from pathlib import Path
import mlflow
import mlflow.spark

PROJECT_ROOT = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project")

TRACKING_URI = f"file:{PROJECT_ROOT / 'mlruns'}"
mlflow.set_tracking_uri(TRACKING_URI)

mlflow.set_experiment("bupa_claims_high_cost")

print("MLflow tracking URI:", mlflow.get_tracking_uri())


MLflow tracking URI: file:/Users/manojrammopati/Public/Projects/bupa_insurance_project/mlruns


# Cell 2 – Imports & config

In [18]:
# Cell 2 — Imports & config

from pyspark.sql import functions as F
from pyspark.sql import DataFrame

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import (
    RandomForestClassifier,
    LogisticRegression,
    GBTClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

ACCOUNT        = "clientdatastorage"
GOLD_CONTAINER = "golddata"

FT_CLAIMS_RISK_SPLIT_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/ft_claims_risk_split"
)

MODEL_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/models"
)
MODEL_NAME = "claims_high_cost_best"
MODEL_PATH = f"{MODEL_BASE_PATH}/{MODEL_NAME}"

print("Feature table path:", FT_CLAIMS_RISK_SPLIT_PATH)
print("Model path        :", MODEL_PATH)


Feature table path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split
Model path        : abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_high_cost_best


# Cell 3 – Load feature table

In [19]:
# Cell 3 — Load ft_claims_risk_split

claims_all = (
    spark.read
         .format("delta")
         .load(FT_CLAIMS_RISK_SPLIT_PATH)
)

print("Total rows in ft_claims_risk_split:", claims_all.count())
claims_all.printSchema()
claims_all.show(5, truncate=False)


Total rows in ft_claims_risk_split: 558211
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Days_To_Settle: integer (nullable = true)
 |-- High_Cost_Claim_Flag: integer (nullable = true)
 |-- Claim_Fraud_Label: integer (nullable = true)
 |-- Provider_Fraud_Label: integer (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_date_valid: integer (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- dataset_split: string (nullable = true)



+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Days_To_Settle|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199

# Cell 4 – Filter label + train/test

In [20]:
# Cell 4 — Use Is_High_Cost as label

LABEL_COL = "Is_High_Cost"

if LABEL_COL not in claims_all.columns:
    raise ValueError(f"Expected label column '{LABEL_COL}' not found in ft_claims_risk_split")

claims_labeled = (
    claims_all
      .filter(F.col(LABEL_COL).isNotNull())
      .withColumn(LABEL_COL, F.col(LABEL_COL).cast("double"))
)

# Optional: keep only dq-valid rows
claims_labeled = claims_labeled.filter(
    (F.col("dq_money_valid") == 1) & (F.col("dq_date_valid") == 1)
)

print("Rows with non-null label & dq-valid:", claims_labeled.count())

train_df = claims_labeled.filter(F.col("dataset_split") == "train")
test_df  = claims_labeled.filter(F.col("dataset_split") == "test")

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())


Rows with non-null label & dq-valid: 558211
Train rows: 446102
Test rows : 112109


# Cell 5 – Feature columns & null handling

In [21]:
# Cell 5 — Feature selection & null handling

numeric_candidates = [
    "Claim_Amount_GBP",
    "Payout_GBP",
    "Payout_to_Amount_Ratio",
    "Days_To_Settle",
]

categorical_candidates = [
    "Claim_Type_Name",
    "Claim_Status",
    "Claim_Type_Code",
    "Provider_Risk_Tier",
    "Provider_ID",
]

cols = claims_labeled.columns

numeric_cols = [c for c in numeric_candidates if c in cols]
cat_cols     = [c for c in categorical_candidates if c in cols]

print("Numeric features   :", numeric_cols)
print("Categorical features:", cat_cols)

id_cols = [c for c in ["Claim_ID", "Member_Key", "Provider_ID"] if c in cols]

def prep_nulls(df: DataFrame) -> DataFrame:
    d = df
    for c in numeric_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    for c in cat_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    return d


Numeric features   : ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle']
Categorical features: ['Claim_Type_Name', 'Claim_Status', 'Claim_Type_Code', 'Provider_Risk_Tier', 'Provider_ID']


# Cell 6 – Build pipeline skeleton

In [22]:
# Cell 6 — Indexers, encoders, assembler

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep",
    )
    for c in cat_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_oh",
        handleInvalid="keep",
    )
    for c in cat_cols
]

feature_cols = numeric_cols + [f"{c}_oh" for c in cat_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep",
)

print("Feature columns in assembler:", feature_cols)


Feature columns in assembler: ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle', 'Claim_Type_Name_oh', 'Claim_Status_oh', 'Claim_Type_Code_oh', 'Provider_Risk_Tier_oh', 'Provider_ID_oh']


# Cell 7 – Train & eval helper

In [23]:
# Cell 7 — Train & evaluate helper

def train_and_eval(pipeline: Pipeline, train_df: DataFrame, test_df: DataFrame, model_name: str):
    print(f"\n===== Training {model_name} =====")
    model = pipeline.fit(train_df)

    print(f"Scoring test set with {model_name} ...")
    pred = model.transform(test_df)

    evaluator_roc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR",
    )

    roc_auc = evaluator_roc.evaluate(pred)
    pr_auc  = evaluator_pr.evaluate(pred)

    preds = pred.select(
        LABEL_COL,
        F.col("prediction").cast("double").alias("prediction")
    )

    tp = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 1)).count()
    tn = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 0)).count()
    fp = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 1)).count()
    fn = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 0)).count()

    total = tp + tn + fp + fn
    accuracy  = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    metrics = {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

    print(f"ROC AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f} | "
          f"Acc: {accuracy:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f}")

    return model, metrics, pred


# Cell 8 – Define models & train

In [24]:
# Cell 8 — Train LR, RF, GBT for high-cost label

train_pre = prep_nulls(train_df)
test_pre  = prep_nulls(test_df)

rf = RandomForestClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=8,
    seed=42,
)

lr = LogisticRegression(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,
)

"""

gbt = GBTClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    maxIter=80,
    maxDepth=6,
    stepSize=0.1,
    seed=42,
)

"""

pipelines = {
    "RandomForest": Pipeline(stages=indexers + encoders + [assembler, rf]),
    "LogisticRegression": Pipeline(stages=indexers + encoders + [assembler, lr]),
    #"GBTClassifier": Pipeline(stages=indexers + encoders + [assembler, gbt]),
}

results = []
trained_models = {}

for name, pipe in pipelines.items():
    model, metrics, _ = train_and_eval(pipe, train_pre, test_pre, name)
    results.append(metrics)
    trained_models[name] = model

print("\n===== Summary of models =====")
for m in results:
    print(m)



===== Training RandomForest =====


25/12/12 15:29:17 WARN MemoryStore: Not enough space to cache rdd_1010_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:29:17 WARN BlockManager: Persisting block rdd_1010_2 to disk instead.
25/12/12 15:29:52 WARN MemoryStore: Not enough space to cache rdd_1010_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:30:20 WARN MemoryStore: Not enough space to cache rdd_1010_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:30:49 WARN MemoryStore: Not enough space to cache rdd_1010_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:31:22 WARN DAGScheduler: Broadcasting large task binary with size 1019.9 KiB
25/12/12 15:31:23 WARN MemoryStore: Not enough space to cache rdd_1010_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:31:55 WARN DAGScheduler: Broadcasting large task binary with size 1059.9 KiB
25/12/12 15:31:55 WARN MemoryStore: Not enough space to cache rdd_1010_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:32:23 WARN DAGScheduler: Broadcasting large task binary 

Scoring test set with RandomForest ...


25/12/12 15:34:03 WARN DAGScheduler: Broadcasting large task binary with size 1062.5 KiB
25/12/12 15:34:08 WARN DAGScheduler: Broadcasting large task binary with size 1062.5 KiB


ROC AUC: 0.9984 | PR AUC: 0.9890 | Acc: 0.8985 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000

===== Training LogisticRegression =====


Scoring test set with LogisticRegression ...


ROC AUC: 0.9988 | PR AUC: 0.9908 | Acc: 0.9644 | Prec: 0.9982 | Rec: 0.6502 | F1: 0.7875

===== Summary of models =====
{'model': 'RandomForest', 'roc_auc': 0.998429285170218, 'pr_auc': 0.9889808658186497, 'accuracy': 0.8984559669607257, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'tp': 0, 'tn': 100725, 'fp': 0, 'fn': 11384}
{'model': 'LogisticRegression', 'roc_auc': 0.9988450363466415, 'pr_auc': 0.9908377343866447, 'accuracy': 0.9643650375973383, 'precision': 0.9982467970330411, 'recall': 0.6502108222066058, 'f1': 0.787488696207245, 'tp': 7402, 'tn': 100712, 'fp': 13, 'fn': 3982}


# Cell 9 – Choose best model & save

In [25]:
# Cell 9 — Select best model (ROC AUC) and save

best = sorted(results, key=lambda r: r["roc_auc"], reverse=True)[0]
best_name = best["model"]
best_model = trained_models[best_name]

print(f"\nBest model by ROC AUC: {best_name} (ROC AUC={best['roc_auc']:.4f})")

best_model.write().overwrite().save(MODEL_PATH)
print(f"✅ Saved best high-cost model to: {MODEL_PATH}")



Best model by ROC AUC: LogisticRegression (ROC AUC=0.9988)


✅ Saved best high-cost model to: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_high_cost_best


# Cell 10 – Load & score sample claims

In [2]:
# Cell 10 — Load saved model and score some claims

loaded_model = PipelineModel.load(MODEL_PATH)

sample_to_score = (
    test_df
        .orderBy(F.rand())
        .limit(20)
)

scored_raw = loaded_model.transform(prep_nulls(sample_to_score))

scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .withColumn("high_cost_prob", F.col("prob_arr")[1])
        .select(
            *[c for c in id_cols if c in scored_raw.columns],
            LABEL_COL,
            "high_cost_prob",
            "prediction",
            "Claim_Type_Name",
            "Claim_Status",
            "Claim_Amount_GBP",
            "Payout_GBP",
            "Payout_to_Amount_Ratio",
            "Provider_Risk_Tier",
        )
)

scored.printSchema()
scored.show(truncate=False)


NameError: name 'PipelineModel' is not defined

#  Cell 11 — MLflow logging for high-cost model

In [1]:


import mlflow
import mlflow.spark

with mlflow.start_run(run_name="high_cost_claims_training") as run:

    # -----------------------------
    # 🔹 Data / label / features
    # -----------------------------
    mlflow.log_param("label_col", LABEL_COL)
    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("test_rows",  test_df.count())
    mlflow.log_param("numeric_features", ",".join(numeric_cols))
    mlflow.log_param("categorical_features", ",".join(cat_cols))

    # -----------------------------
    # 🔹 Metrics for each model
    # -----------------------------
    for m in results:
        model_name = m["model"]
        prefix = model_name.replace(" ", "_")

        mlflow.log_metric(f"{prefix}_roc_auc",   float(m["roc_auc"]))
        mlflow.log_metric(f"{prefix}_pr_auc",    float(m["pr_auc"]))
        mlflow.log_metric(f"{prefix}_accuracy",  float(m["accuracy"]))
        mlflow.log_metric(f"{prefix}_precision", float(m["precision"]))
        mlflow.log_metric(f"{prefix}_recall",    float(m["recall"]))
        mlflow.log_metric(f"{prefix}_f1",        float(m["f1"]))

        # confusion matrix bits
        mlflow.log_metric(f"{prefix}_tp", float(m["tp"]))
        mlflow.log_metric(f"{prefix}_tn", float(m["tn"]))
        mlflow.log_metric(f"{prefix}_fp", float(m["fp"]))
        mlflow.log_metric(f"{prefix}_fn", float(m["fn"]))

    # -----------------------------
    # 🔹 Best model details
    # -----------------------------
    mlflow.log_param("best_model", best_name)

    if best_name == "RandomForest":
        mlflow.log_param("algo", "RandomForest")
        mlflow.log_param("numTrees", rf.getNumTrees())
        mlflow.log_param("maxDepth", rf.getOrDefault("maxDepth"))
    elif best_name == "LogisticRegression":
        mlflow.log_param("algo", "LogisticRegression")
        mlflow.log_param("maxIter", lr.getOrDefault("maxIter"))
        mlflow.log_param("regParam", lr.getOrDefault("regParam"))

    # -----------------------------
    # 🔹 Log Spark model artifact
    # -----------------------------
    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="high_cost_best_model",
        registered_model_name=None,  # can register later if you want
    )

    run_id = run.info.run_id

print("✅ MLflow logging completed for high-cost model. Run ID:", run_id)


2025/12/12 16:25:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/12 16:25:20 INFO mlflow.store.db.utils: Updating database tables
2025/12/12 16:25:20 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/12 16:25:20 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/12 16:25:20 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/12 16:25:20 INFO alembic.runtime.migration: Will assume non-transactional DDL.


NameError: name 'LABEL_COL' is not defined

# Cell 12 Model Registration in MLFLOW

from mlflow.tracking import MlflowClient

client = MlflowClient()

model_uri = f"runs:/{run_id}/high_cost_best_model"

try:
    client.create_registered_model("bupa_high_cost_model")
except Exception as e:
    print("Registered model may already exist:", e)

client.create_model_version(
    name="bupa_high_cost_model",
    source=model_uri,
)

print("📌 Registered new model version for bupa_high_cost_model from run:", run_id)


# 📈 Cell 13 SAVING ALL MODELS METRICS TO ML_MONITORING

In [28]:
from datetime import datetime
import utils_silver

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(ACCOUNT)

now_ts = datetime.utcnow()
rows = []

for m in results:
    rows.append({
        "model_name":   m["model"],
        "use_case":     "high_cost_claims",
        "dataset_name": "ft_claims_risk",
        "dataset_split": "test",
        "auc":       float(m.get("roc_auc", m.get("auc_roc", 0.0))),
        "accuracy":  float(m.get("accuracy", 0.0)),
        "precision": float(m.get("precision", 0.0)),
        "recall":    float(m.get("recall", 0.0)),
        "f1":        float(m.get("f1", 0.0)),
        "ts":        now_ts,
        "notes":     "high-cost model sweep",
    })

utils_silver.write_ml_monitoring(
    spark=spark,
    rows=rows,
    paths_gold=paths_gold,
)


[ML_MON] Wrote 2 rows to abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring
